# Holiday Package Prediction 

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline


In [2]:
df=pd.read_csv("Travel.csv")
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [3]:
df.shape

(4888, 20)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                4888 non-null   int64  
 1   ProdTaken                 4888 non-null   int64  
 2   Age                       4662 non-null   float64
 3   TypeofContact             4863 non-null   object 
 4   CityTier                  4888 non-null   int64  
 5   DurationOfPitch           4637 non-null   float64
 6   Occupation                4888 non-null   object 
 7   Gender                    4888 non-null   object 
 8   NumberOfPersonVisiting    4888 non-null   int64  
 9   NumberOfFollowups         4843 non-null   float64
 10  ProductPitched            4888 non-null   object 
 11  PreferredPropertyStar     4862 non-null   float64
 12  MaritalStatus             4888 non-null   object 
 13  NumberOfTrips             4748 non-null   float64
 14  Passport

### Data cleaning


In [6]:
df.isnull().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [7]:
df['Gender'].value_counts()

Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64

In [8]:
df['Occupation'].value_counts()

Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64

In [9]:
df['MaritalStatus'].value_counts()

MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64

In [10]:
df['TypeofContact'].value_counts()

TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64

In [11]:
df['ProductPitched'].value_counts()

ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64

In [12]:
df['Gender']=df['Gender'].replace('Fe Male','Female')
df['MaritalStatus']=df['MaritalStatus'].replace('Single','Unmarried')

In [13]:
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Unmarried,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [14]:
feature_with_na=[features for features in df.columns if df[features].isnull().sum()>=1]
for feature in feature_with_na:
    print(feature,np.round(df[feature].isnull().mean()*100,5),'% missing values')

Age 4.62357 % missing values
TypeofContact 0.51146 % missing values
DurationOfPitch 5.13502 % missing values
NumberOfFollowups 0.92062 % missing values
PreferredPropertyStar 0.53191 % missing values
NumberOfTrips 2.86416 % missing values
NumberOfChildrenVisiting 1.35025 % missing values
MonthlyIncome 4.76678 % missing values


In [15]:
df[feature_with_na].select_dtypes(exclude='object').describe()

,Age,DurationOfPitch,NumberOfFollowups,PreferredPropertyStar,NumberOfTrips,NumberOfChildrenVisiting,MonthlyIncome
count,4662.000000,4637.000000,4843.000000,4862.000000,4748.000000,4822.000000,4655.000000
mean,37.622265,15.490835,3.708445,3.581037,3.236521,1.187267,23619.853491
std,9.316387,8.519643,1.002509,0.798009,1.849019,0.857861,5380.698361
min,18.000000,5.000000,1.000000,3.000000,1.000000,0.000000,1000.000000
25%,31.000000,9.000000,3.000000,3.000000,2.000000,1.000000,20346.000000
50%,36.000000,13.000000,4.000000,3.000000,3.000000,1.000000,22347.000000
75%,44.000000,20.000000,4.000000,4.000000,4.000000,2.000000,25571.000000
max,61.000000,127.000000,6.000000,5.000000,22.000000,3.000000,98678.000000


In [16]:
df.Age.fillna(df.Age.median(),inplace=True)
df.TypeofContact.fillna(df.TypeofContact.mode()[0],inplace=True)
df.DurationOfPitch.fillna(df.DurationOfPitch.mode()[0],inplace=True)
df.NumberOfFollowups.fillna(df.NumberOfFollowups.mode()[0],inplace=True)
df.PreferredPropertyStar.fillna(df.PreferredPropertyStar.mode()[0],inplace=True)
df.NumberOfTrips.fillna(df.NumberOfTrips.median(),inplace=True)
df.NumberOfChildrenVisiting.fillna(df.NumberOfChildrenVisiting.mode()[0],inplace=True)
df.MonthlyIncome.fillna(df.MonthlyIncome.median(),inplace=True)



In [17]:
df.isnull().sum()

CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

In [18]:
df.drop('CustomerID',inplace=True,axis=1)

In [19]:
# feature extraction
df['Totalvisiting']=df['NumberOfPersonVisiting']+df['NumberOfChildrenVisiting']
df.drop(columns=['NumberOfChildrenVisiting','NumberOfPersonVisiting'],axis=1,inplace=True)

In [20]:
### get all the numeric feature 
num_features=[feature for feature in df.columns if df[feature].dtype!='O']
print('Num of Numerical feature:',len(num_features))

Num of Numerical feature: 12


In [21]:
cat_feature=[feature for feature in df.columns if df[feature].dtype=='O']
print("numer of categorical feature:",len(cat_feature))

numer of categorical feature: 6


In [22]:
discrete_feature=[feature for feature in num_features if len(df[feature].unique())<=25]
print("num of discrete feature:",len(discrete_feature))

num of discrete feature: 9


In [23]:
continous_feature=[feature for feature in num_features if len(df[feature].unique())>25]
print("num of discrete feature:",len(continous_feature))

num of discrete feature: 3


In [29]:
target_corr=df.corr(numeric_only=True)["ProdTaken"].sort_values(ascending=False)
print(target_corr)


ProdTaken                 1.000000
Passport                  0.260844
NumberOfFollowups         0.111560
PreferredPropertyStar     0.098704
CityTier                  0.086852
DurationOfPitch           0.076089
PitchSatisfactionScore    0.051394
NumberOfTrips             0.019709
Totalvisiting             0.009759
OwnCar                   -0.011508
MonthlyIncome            -0.126420
Age                      -0.143753
Name: ProdTaken, dtype: float64


In [24]:
from sklearn.model_selection import train_test_split
X=df.drop(["ProdTaken"],axis=1)
y=df['ProdTaken']

In [25]:
X.shape

(4888, 17)

In [26]:
y.value_counts()

ProdTaken
0    3968
1     920
Name: count, dtype: int64

In [27]:
# train test split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape,X_test.shape

((3910, 17), (978, 17))

In [28]:
#Create Column transformer with 3 type of transformers
cat_feature=X.select_dtypes(include="object").columns
num_features=X.select_dtypes(exclude="object").columns

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer=StandardScaler()
oh_transformer=OneHotEncoder(drop="first")

preprocessor=ColumnTransformer(
    [("One Hot Encoder",oh_transformer,cat_feature),
    ("StandardScaler",numeric_transformer,num_features)
     
     ]
)

In [29]:
preprocessor

,transformers,"[('One Hot Encoder', ...), ('StandardScaler', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,True


In [30]:
X_train=preprocessor.fit_transform(X_train)

In [31]:
X_test=preprocessor.transform(X_test)

In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn .metrics import accuracy_score,classification_report,confusion_matrix,precision_score,recall_score, \
                            f1_score,roc_auc_score,roc_curve

In [33]:
models={
    "Decision tree":DecisionTreeClassifier(),
    "Random forest":RandomForestClassifier()

}
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train)#train model

    #make prediction
    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)

    #trainig set performance
    model_train_accuracy=accuracy_score(y_train,y_train_pred)
    model_train_f1=f1_score(y_train,y_train_pred,average="weighted")
    model_train_precision=precision_score(y_train,y_train_pred)
    model_train_recall=recall_score(y_train,y_train_pred)
    model_train_rocauc_curve=roc_auc_score(y_train,y_train_pred)


    #test set performance
    model_test_accuracy=accuracy_score(y_test,y_test_pred)
    model_test_f1=f1_score(y_test,y_test_pred,average="weighted")
    model_test_precision=precision_score(y_test,y_test_pred)
    model_test_recall=recall_score(y_test,y_test_pred)
    model_test_rocauc_curve=roc_auc_score(y_test,y_test_pred)

    print(list(models.keys())[i])

    print("Model performance for trainig set")
    print("-accuracy:{:.4f}".format(model_train_accuracy))
    print("-F1 score:{:.4f}".format(model_train_f1))

    print("-preciion:{:.4f}".format(model_train_precision))
    print("-recall:{:.4f}".format(model_train_recall))
    print("-roc auc score:{:.4f}".format(model_train_rocauc_curve))

    print("----------------------------------")
    print("Model performance for test set")
    print("-accuracy:{:.4f}".format(model_test_accuracy))
    print("-F1 score:{:.4f}".format(model_test_f1))

    print("-preciion:{:.4f}".format(model_test_precision))
    print("-recall:{:.4f}".format(model_test_recall))
    print("-roc auc score:{:.4f}".format(model_test_rocauc_curve))
    print("----------------------------------")







Decision tree
Model performance for trainig set
-accuracy:1.0000
-F1 score:1.0000
-preciion:1.0000
-recall:1.0000
-roc auc score:1.0000
----------------------------------
Model performance for test set
-accuracy:0.9151
-F1 score:0.9140
-preciion:0.8034
-recall:0.7487
-roc auc score:0.8521
----------------------------------
Random forest
Model performance for trainig set
-accuracy:1.0000
-F1 score:1.0000
-preciion:1.0000
-recall:1.0000
-roc auc score:1.0000
----------------------------------
Model performance for test set
-accuracy:0.9254
-F1 score:0.9199
-preciion:0.9470
-recall:0.6545
-roc auc score:0.8228
----------------------------------


In [34]:
#hyperparameter tuning

rf_params={
    "max_depth":[5,8,15,None,10],
    "max_features":[5,7,'auto',8],
    "min_samples_split":[2,8,15,20],
    "n_estimators":[100,200,500,1000]
}

In [35]:
# model list for hyperparametric tuning
randomcv_models=[
    ("RF",RandomForestClassifier(),rf_params)
]

In [36]:
from sklearn.model_selection import RandomizedSearchCV

model_params={}
for name ,model, params in randomcv_models:
    random=RandomizedSearchCV(estimator=model,param_distributions=params,n_iter=100,cv=3,verbose=2,n_jobs=-1)
    random.fit(X_train,y_train)
    model_params[name]=random.best_params_

for model_name in model_params:
    print(f"---------best params for {model_name}-------")
    print(model_params[model_name])

Fitting 3 folds for each of 100 candidates, totalling 300 fits
---------best params for RF-------
{'n_estimators': 1000, 'min_samples_split': 2, 'max_features': 8, 'max_depth': None}


In [37]:
models={
    "Random forest":RandomForestClassifier(n_estimators=1000, min_samples_split= 2, max_features=8, max_depth= None)

}
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train)#train model

    #make prediction
    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)

    #trainig set performance
    model_train_accuracy=accuracy_score(y_train,y_train_pred)
    model_train_f1=f1_score(y_train,y_train_pred,average="weighted")
    model_train_precision=precision_score(y_train,y_train_pred)
    model_train_recall=recall_score(y_train,y_train_pred)
    model_train_rocauc_curve=roc_auc_score(y_train,y_train_pred)


    #test set performance
    model_test_accuracy=accuracy_score(y_test,y_test_pred)
    model_test_f1=f1_score(y_test,y_test_pred,average="weighted")
    model_test_precision=precision_score(y_test,y_test_pred)
    model_test_recall=recall_score(y_test,y_test_pred)
    model_test_rocauc_curve=roc_auc_score(y_test,y_test_pred)

    print(list(models.keys())[i])

    print("Model performance for trainig set")
    print("-accuracy:{:.4f}".format(model_train_accuracy))
    print("-F1 score:{:.4f}".format(model_train_f1))

    print("-preciion:{:.4f}".format(model_train_precision))
    print("-recall:{:.4f}".format(model_train_recall))
    print("-roc auc score:{:.4f}".format(model_train_rocauc_curve))

    print("----------------------------------")
    print("Model performance for test set")
    print("-accuracy:{:.4f}".format(model_test_accuracy))
    print("-F1 score:{:.4f}".format(model_test_f1))

    print("-preciion:{:.4f}".format(model_test_precision))
    print("-recall:{:.4f}".format(model_test_recall))
    print("-roc auc score:{:.4f}".format(model_test_rocauc_curve))
    print("----------------------------------")







Random forest
Model performance for trainig set
-accuracy:1.0000
-F1 score:1.0000
-preciion:1.0000
-recall:1.0000
-roc auc score:1.0000
----------------------------------
Model performance for test set
-accuracy:0.9387
-F1 score:0.9350
-preciion:0.9645
-recall:0.7120
-roc auc score:0.8528
----------------------------------


In [38]:
import pickle
pickle.dump(model,open('best_model.pkl','wb'))